## Importing Libraries

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

load_dotenv()

llm = ChatOpenAI(model="gpt-5-nano", temperature=0.0)

## 1. SIMPLE CHAIN 

In [4]:
print("=== Simple Chain ===")
prompt = ChatPromptTemplate.from_template(
    "Summarize {concept} for a backend engineer in exactly 2 sentences."
)

chain = prompt | llm | StrOutputParser()

result = chain.invoke({"concept": "redis pub/sub"})
print(result)
print()

=== Simple Chain ===
Redis Pub/Sub provides a lightweight, real-time messaging model where publishers emit messages to named channels and all currently connected subscribers receive them immediately. Messages are not persisted or durably queued, so delivery is not guaranteed and late subscribers won’t see past events; for durability and replayability, consider Redis Streams or another queuing mechanism.



## 2. MULTI-STEP CHAIN 
*Step 1: explain technically*

*Step 2: take that explanation and simplify it*

*Wire them: explanation from chain 1 becomes input to chain 2*

In [5]:
print("=== Multi-Step Chain ===")

explain_prompt = ChatPromptTemplate.from_template(
    "Explain {concept} in technical depth. 1 sentence."
)

explain_chain = explain_prompt | llm | StrOutputParser()

simplify_prompt = ChatPromptTemplate.from_template(
    "Simplify this for a junior engineer in 1 sentence: {explanation}"
)
simplify_chain = simplify_prompt | llm | StrOutputParser()

full_chain = {"explanation": explain_chain} | simplify_chain

result = full_chain.invoke({"concept": "database connection pooling"})
print(result)
print()

=== Multi-Step Chain ===
A connection pool keeps a limited set of open database connections and reuses them for requests to avoid reconnecting each time, creates new ones up to a configured max, returns idle connections to the pool rather than closing them, validates and evicts broken ones, optionally caches prepared statements and detects leaks, enforces lifecycle rules like timeouts and max lifetimes, and is thread-safe with hooks for transactions, failover, and monitoring.



## 3. RUNNABLEPASSTHROUGH — preserve original input

*.assign() adds new keys to the input dict without removing existing ones*

In [6]:
print("=== RunnablePassthrough ===")

chain_with_passthrough = RunnablePassthrough.assign(
    summary = prompt | llm | StrOutputParser()
)

result = chain_with_passthrough.invoke({"concept": "JWT authentication"})
print(f"Original summary: {result['concept']}")
print(f"Passthrough summary: {result['summary']}")
print()

=== RunnablePassthrough ===
Original summary: JWT authentication
Passthrough summary: JWT authentication issues a token signed by the server (symmetric HS256 or asymmetric RS256/ES256) that carries claims such as user_id and roles, and the client sends it in the Authorization: Bearer header on each request. The backend validates the token by verifying its signature and claims (exp, iss, aud), optionally fetching public keys via JWKS, and uses short-lived access tokens with refresh tokens to stay stateless and scalable.



## 4. RUNNABLELAMBDA — wrap any function

In [7]:
print("=== RunnableLambda ===")
def preprocess_input(input_dict: dict) -> dict:
    """Normalize and enrich the input before sending to LLM."""
    input_dict["concept"] = input_dict["concept"].strip().title()
    input_dict["audience"] = "senior backend engineer"
    return input_dict

def postprocess_output(output: str) -> str:
    """Format the output after receiving from LLM."""
    lines = [line.strip() for line in output.split("\n") if line.strip()]
    return "\n".join(f" → {line}" for line in lines)

audience_prompt = ChatPromptTemplate.from_template(
    "Explain {concept} to a {audience}."
)

chain_with_lambda = RunnableLambda(preprocess_input) | audience_prompt | llm | StrOutputParser() | RunnableLambda(postprocess_output)

result = chain_with_lambda.invoke({"concept": "database sharding"})
print(result)
print()

=== RunnableLambda ===
 → Sharding is horizontal partitioning of a database: you split a large dataset across multiple physical servers (shards) so each shard holds only a portion of the data. The application (or a routing layer) knows how to route a query to the right shard(s) based on the shard key. The goal is to scale writes and storage, improve latency by geographic locality, and often isolate tenants or data domains.
 → Key concepts senior engineers care about
 → - Why shard: to absorb higher write throughput, larger data volume, and potentially multi-tenant isolation. It trades some complexity and operational overhead for scale and resilience.
 → - Horizontal vs vertical sharding:
 → - Horizontal sharding: partition rows across multiple shards; the canonical form of “sharding.”
 → - Vertical partitioning: move different tables or functional components to different servers. It’s not sharding in the strict sense, but is often used to separate hot vs cold data or read-heavy vs writ

## 5. STREAMING 

In [8]:
print("=== Streaming ===")
stream_prompt = ChatPromptTemplate.from_template(
    "List 3 key concepts every backend engineer must know. Be detailed."
)

stream_chain = stream_prompt | llm | StrOutputParser()

print("Streaming output:", end=" ")
for chunk in stream_chain.stream({}):
    print(chunk, end=" ", flush=True)

print("\nDone streaming.")

=== Streaming ===
Streaming output:  Here  are  three  foundational  concepts  every  backend  engineer  should  know ,  with  detail  on  what  they  involve  and  how  to  apply  them  in  real  systems .

 1 )  Data  storage  and  data  modeling  ( p ersistence ,  integrity ,  and  access  patterns )
 What  it  covers 
 -  Choosing  the  right  storage :  relational  ( SQL )  vs  non -rel ational  ( No SQL )  vs  specialized  stores  ( column ar ,  time -series ,  graph ).
 -  Data  modeling  decisions :  normalization  vs  den ormal ization ,  how  you  structure  tables / collections ,  and  how  that  matches  query  patterns .
 -  Storage  guarantees :  AC ID  transactions  vs  BASE / event ual  consistency ,  isolation  levels ,  and  how  consistency  affects  correctness .
 -  Index ing  and  query  optimization :  creating  the  right  indexes  for  common  queries ,  avoiding  hotspots ,  and  understanding  query  plans .
 -  M igrations  and  schema  evolution :  how  to 

## 6. BATCH — run multiple inputs in parallel 

In [9]:
print("=== Batch ===")
batch_chain = prompt | llm | StrOutputParser()

results = batch_chain.batch([
    {"concept": "PostgreSQL MVCC"},
    {"concept": "Redis eviction policies"},
    {"concept": "JWT token refresh"},
])
print("Batch results:")
for i, result in enumerate(results, 1):
    print(f"  {i}. {result[:80]}")

print()

=== Batch ===
Batch results:
  1. PostgreSQL uses MVCC to provide each transaction a consistent snapshot of the da
  2. Redis eviction policies determine which keys to drop when maxmemory is reached a
  3. Use short‑lived access tokens (e.g., 15 minutes) with long‑lived refresh tokens 

